# 01 · LFW 데이터 준비
고정된 6,000 pair 목록과 이미지 대응, PQ development 인물 분리를 검증합니다.
기존 image manifest를 재사용합니다. 새 환경에서만 첫 셀의 생성 옵션을 켭니다.
설정 확인 후 Kernel Restart → Run All. [실행 안내](README.md) · [지표·보존 안내](LFW_PAIR_VERIFICATION.md)

In [ ]:
CONFIG_PATH = "configs/experiments/lfw_pair_verification.yaml"
CREATE_IMAGE_MANIFEST = False  # 파일이 없을 때만 생성; 기존 입력은 덮어쓰지 않음
DOWNLOAD_LFW_PAIRS = False  # 고정 목록이 없는 최초 실행에서만 True
LFW_ROOT = "data/raw/LFW/lfw-deepfunneled/lfw-deepfunneled"

In [ ]:
from pathlib import Path
import sys
import yaml
from IPython.display import display

PROJECT_ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                    if (p / "research").is_dir() and (p / "configs").is_dir())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("Python:", sys.executable)

import pandas as pd
from research.datasets import build_lfw_manifest, write_lfw_manifest_bundle
from research.datasets.lfw_pairs import load_pairs
config = yaml.safe_load((PROJECT_ROOT / CONFIG_PATH).read_text(encoding="utf-8"))
manifest_path = PROJECT_ROOT / config["image_manifest"]
if not manifest_path.is_file():
    if not CREATE_IMAGE_MANIFEST:
        raise FileNotFoundError(f"{manifest_path}: 첫 셀에서 CREATE_IMAGE_MANIFEST=True로 설정하세요.")
    if manifest_path.name != "face_manifest.csv":
        raise ValueError("image_manifest 파일명은 face_manifest.csv여야 합니다.")
    bundle = build_lfw_manifest(PROJECT_ROOT / LFW_ROOT, PROJECT_ROOT)
    write_lfw_manifest_bundle(bundle, manifest_path.parent, overwrite=False)
pairs, development = load_pairs(PROJECT_ROOT / config["pairs_file"],
    pd.read_csv(manifest_path), download=DOWNLOAD_LFW_PAIRS)

In [ ]:
display(pairs.groupby(["fold", "is_genuine"]).size().rename("pair_count").reset_index())
print("Test pairs:", len(pairs))
print("PQ development candidate pool (before held-out people exclusion):", len(development), "images /", development.identity_id.nunique(), "identities")
print("다음: 02_inputs.ipynb — 같은 CONFIG_PATH를 사용하세요.")
if config.get("pq_training_policy") == "outer_fold_unused_identities":
    from research.datasets.lfw_pairs import outer_fold_development
    fold_development = outer_fold_development(PROJECT_ROOT / config["people_file"],
        pd.read_csv(manifest_path), pairs, development)
    display(pd.DataFrame([dict(fold=f, pq_images=len(d), pq_identities=d.identity_id.nunique())
                          for f, d in fold_development.items()]))


10 folds 각각 genuine 300쌍 + impostor 300쌍을 유지합니다. PQ development는 모든 pair에 등장하는 인물을 제외합니다.
image manifest의 기존 split 열은 pair 역할 결정에 사용하지 않습니다. FR/FIQA 추출은 다음 단계에서 수행합니다.